In [1]:
# Torch version
!python -c "import torch; print(torch.__version__)"

# Cuda version
!python -c "import torch; print(torch.version.cuda)"

2.6.0+cu124
12.4


In [2]:
# Uninstall
# !pip uninstall torch-scatter torch-sparse torch-cluster torch-spline-conv pyg-lib -y

In [3]:
# Update Torch
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [4]:
# Install PyG (automatic)
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch_geometric

In [5]:
# Verify instalation
import torch
import torch_geometric
import torch_scatter

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch_scatter.__version__)
print(torch_geometric.__version__)


2.6.0+cu124
12.4
True
2.1.2+pt26cu124
2.7.0


In [6]:
from model_PyG import *
from utils import *

In [7]:
import json
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T

from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.transforms import Compose
from torch_geometric.utils import dense_to_sparse, negative_sampling
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.optim import Adam

In [8]:
import torch_geometric
print(torch_geometric.__version__)

np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

2.7.0


### Utils

In [9]:
def info(data):
	print("Validate:\t {}".format(data.validate(raise_on_error=True)))
	print("Num. nodes:\t {}".format(data.num_nodes))
	print("Num. edges:\t {}".format(data.num_edges))
	print("Num. features:\t {}".format(data.num_node_features))
	print("Has isolated:\t {}".format(data.has_isolated_nodes()))
	print("Has loops:\t {}".format(data.has_self_loops()))
	print("Is directed:\t {}".format(data.is_directed()))
	print("Is undirected:\t {}".format(data.is_undirected()))
	print("{}".format(data.edge_index))
	print("{}".format(data.x))
	print("{}".format(data.edge_attr))

def compute_num_neg_samples(edge_index, num_nodes, ratio):
	E = edge_index.size(1)
	max_neg = num_nodes * num_nodes - E
	return min(int(ratio * E), max_neg)

def neg_ratio_schedule(epoch, max_epoch):
	start = 5.0
	end = 1.0
	return start - (start - end) * (epoch / max_epoch)

class EarlyStopping:
	def __init__(self, patience=5, delta=0, warmup=5, verbose=False):
		self.patience = patience
		self.delta = delta
		self.warmup = warmup
		self.verbose = verbose
		self.best_loss = None
		self.no_improvement_count = 0
		self.stop_training = False
	
	def check_early_stop(self, loss, epoch):
		if epoch >= self.warmup:
			if self.best_loss is None or loss < self.best_loss - self.delta:
				self.best_loss = loss
				self.no_improvement_count = 0
			else:
				self.no_improvement_count += 1
				if self.no_improvement_count >= self.patience:
					self.stop_training = True
					if self.verbose:
						print("Stopping early as no improvement has been observed.")

### Parameters

In [10]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"] # Change to static, e.g. "exp1"

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

apply_transformation = params["apply_transformation"]
print("Has transformation:", apply_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t", cuda)

epochs = params["epochs"]
print("Epochs:\t", epochs)

lr = params["lr"]
print("Lr:\t", lr)

Exp:		 exp100
Raw data:	 ACM_DBLP
Methods:	 ['t-gae']
Has transformation: False
Dimension:	 128
Groups id:	 ['ACM-DBLP']
Subgroups id:	 {'ACM-DBLP': ['1', '2']}
Cuda:	 1
Epochs:	 500
Lr:	 0.0001


### Setup

In [11]:
# Parameters models

dataset = exp
encoders = ["GIN", "GINE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # Change
device = torch.device(f"cuda:{cuda}" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

NUM_HIDDEN_LAYERS = 12
HIDDEN_DIM = [1024] * NUM_HIDDEN_LAYERS + [1024]
output_feature_size = dimension # 128
# lr = 0.001
# epochs = 100

In [12]:
# Create datasets (groups and subgroups)

list_train_set = []

for group_id in groups_id:
	train_set = []
	for subgroup_id in subgroups_id[group_id]:
		train_set.append("{}_{}".format(group_id, subgroup_id))
	list_train_set.append(train_set)
list_train_set

[['ACM-DBLP_1', 'ACM-DBLP_2']]

### Create Data (PyG)

In [13]:
def add_edge_attributes(data: Data) -> Data:
	"""
	Compute edge attributes:
		1. Common Neighbors
		2. Jaccard Similarity
		3. Adamic-Adar
		4. Feature Similarity (cosine)

	The resulting edge_attr has shape [num_edges, 4].
	"""

	edge_index = data.edge_index
	x = data.x

	num_nodes = data.num_nodes
	num_edges = edge_index.size(1)

	# ---------------------------------------------------------
	# 1. Build an undirected NetworkX graph
	# ---------------------------------------------------------
	G = nx.Graph()
	G.add_nodes_from(range(num_nodes))

	edges = edge_index.t().tolist()

	# Remove self-loops and duplicate edges
	G.add_edges_from(
		(u, v) for u, v in edges if u != v
	)

	# ---------------------------------------------------------
	# 2. Compute node degrees
	# ---------------------------------------------------------
	degree = dict(G.degree())

	# ---------------------------------------------------------
	# 3. Compute feature similarity
	# ---------------------------------------------------------
	if x is not None:

		# Normalize node feature vectors
		x_norm = F.normalize(x.float(), p=2, dim=1)

		# Cosine similarity for each edge
		u = edge_index[0]
		v = edge_index[1]

		feature_sim = (x_norm[u] * x_norm[v]).sum(dim=1)

	else:
		feature_sim = torch.zeros(
			num_edges,
			dtype=torch.float
		)

	# ---------------------------------------------------------
	# 4. Compute structural edge attributes
	# ---------------------------------------------------------
	cn_values = []
	jaccard_values = []
	aa_values = []

	for u, v in edges:

		# Self-loops
		if u == v:
			cn = 0.0
			jaccard = 0.0
			aa = 0.0

		else:

			# Common neighbors
			common = set(nx.common_neighbors(G, u, v))
			cn = float(len(common))

			# Jaccard similarity
			neighbors_u = set(G.neighbors(u))
			neighbors_v = set(G.neighbors(v))

			union = neighbors_u | neighbors_v

			if len(union) > 0:
				jaccard = len(common) / len(union)
			else:
				jaccard = 0.0

			# Adamic-Adar
			aa = 0.0

			for z in common:
				deg_z = degree[z]

				if deg_z > 1:
					aa += 1.0 / torch.log(
						torch.tensor(float(deg_z))
					).item()

		cn_values.append(cn)
		jaccard_values.append(jaccard)
		aa_values.append(aa)

	# ---------------------------------------------------------
	# 5. Convert structural attributes to tensors
	# ---------------------------------------------------------
	cn = torch.tensor(
		cn_values,
		dtype=torch.float
	)

	jaccard = torch.tensor(
		jaccard_values,
		dtype=torch.float
	)

	adamic_adar = torch.tensor(
		aa_values,
		dtype=torch.float
	)

	# ---------------------------------------------------------
	# 6. Combine all edge attributes
	# ---------------------------------------------------------
	edge_attr = torch.stack(
		[
			cn,
			jaccard,
			adamic_adar,
			feature_sim
		],
		dim=1
	)

	# ---------------------------------------------------------
	# 7. Store in the PyG Data object
	# ---------------------------------------------------------
	data.edge_attr = edge_attr

	return data

def fit_edge_normalization(data1, data2, eps=1e-8):

	edge_attr = torch.cat(
		[data1.edge_attr, data2.edge_attr],
		dim=0
	).float()

	mean = edge_attr.mean(dim=0, keepdim=True)
	std = edge_attr.std(dim=0, keepdim=True)

	std = std.clamp_min(eps)

	return mean, std

def apply_edge_normalization(data, mean, std):

	data.edge_attr = (
		data.edge_attr.float() - mean
	) / std

	return data

In [14]:
def NormalizeNodeEdge(data1, data2):
	node_scaler = StandardScaler() # StandardScaler(with_mean=False)
	edge_scaler = StandardScaler() # StandardScaler(with_mean=False)

	# -------------------------
	# Node features
	# -------------------------
	x_all = np.vstack([
		data1.x.cpu().numpy(),
		data2.x.cpu().numpy()
	])

	node_scaler.fit(x_all)

	data1.x = torch.tensor(
		node_scaler.transform(data1.x.cpu().numpy()),
		dtype=torch.float
	)

	data2.x = torch.tensor(
		node_scaler.transform(data2.x.cpu().numpy()),
		dtype=torch.float
	)

	# -------------------------
	# Edge attributes
	# -------------------------
	edge_all = np.vstack([
		data1.edge_attr.cpu().numpy(),
		data2.edge_attr.cpu().numpy()
	])

	edge_scaler.fit(edge_all)

	data1.edge_attr = torch.tensor(
		edge_scaler.transform(data1.edge_attr.cpu().numpy()),
		dtype=torch.float
	)

	data2.edge_attr = torch.tensor(
		edge_scaler.transform(data2.edge_attr.cpu().numpy()),
		dtype=torch.float
	)

In [15]:
# Only for GIN
""" transform = Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(fill_value=1.0),
	T.ToDevice(device)
]) """

# For GIN and GINE
transform = T.Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(attr="edge_attr", fill_value="mean"),
	# T.AddLaplacianEigenvectorPE(k=3, attr_name=None, is_undirected=True),
	# T.AddRandomWalkPE(walk_length=8, attr_name=None),
	T.ToDevice(device)
])

In [16]:

print("Loading training datasets")

list_train_loader = []

if raw_data_file == "ACM_DBLP":
	group_id = groups_id[0]
	
	# Read dataset
	b = np.load("data/ACM-DBLP.npz")

	# Node matching
	test_pairs = b["test_pairs"].astype(np.int32)
	pos_pairs = b["pos_pairs"].astype(np.int32)
	df_node_alignment_truth = pd.DataFrame(np.concatenate((test_pairs, pos_pairs), axis=0))
	df_node_alignment_truth

	# Create data
	subgroup_id = 1
	edge_index1 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x1 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

	subgroup_id = 2
	edge_index2 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x2 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

elif raw_data_file == "Douban Online_Offline":
	group_id = groups_id[0]
	
	# Read dataset
	a1, f1, a2, f2, test_pairs = load_douban()

	# Node matching
	test_pairs = torch.tensor(np.array(test_pairs, dtype=int)) - 1
	test_pairs = test_pairs.numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs.T)
	df_node_alignment_truth

	# Create data
	edge_index1, _ = dense_to_sparse(torch.from_numpy(a1.toarray()))
	x1 = torch.from_numpy(f1.toarray()).float()

	edge_index2, _= dense_to_sparse(torch.from_numpy(a2.toarray()))
	x2 = torch.from_numpy(f2.toarray()).float()

elif raw_data_file == "Cora1-Cora2":
	group_id = groups_id[0]
		
	# Read dataset
	loader = load_npz("data/cora.npz")
	data = loader["adj_matrix"]
	samples = data.shape[0]
	features = data.shape[1]
	values = data.data
	coo_data = data.tocoo()
	# indices = torch.LongTensor([coo_data.row, coo_data.col])
	indices = torch.from_numpy(np.array([coo_data.row, coo_data.col]))

	# Node matching
	N = 2708
	test_pairs = [[i, i] for i in range(N)]
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	node_features = loader["node_attr"]
	node_features = node_features.toarray()

	edge_index1 = torch.tensor(indices, dtype=torch.long)
	x1 = torch.tensor(node_features, dtype=torch.float)

	edge_index2 = torch.tensor(indices, dtype=torch.long)
	x2 = torch.tensor(node_features, dtype=torch.float)

# Create data
train_loader = {}

subgroup_id = 1
data1 = Data(x=x1, edge_index=edge_index1)
data1 = add_edge_attributes(data1)
print(subgroup_id)
info(data1)

subgroup_id = 2
data2 = Data(x=x2, edge_index=edge_index2)
data2 = add_edge_attributes(data2)
print(subgroup_id)
info(data2)

# Normalize
# NormalizeNodeEdge(data1, data2)

subgroup_id = 1
data1 = transform(data1)
train_loader[f"{group_id}_{subgroup_id}"] = data1
print(subgroup_id)
info(data1)

subgroup_id = 2
data2 = transform(data2)
train_loader[f"{group_id}_{subgroup_id}"] = data2
print(subgroup_id)
info(data2)

list_train_loader.append(train_loader)

Loading training datasets
1
Validate:	 True
Num. nodes:	 9872
Num. edges:	 79122
Num. features:	 17
Has isolated:	 False
Has loops:	 False
Is directed:	 False
Is undirected:	 True
tensor([[   0,    0,    0,  ..., 9870, 9870, 9871],
        [   1,    2,    3,  ..., 1586, 2482, 4454]])
tensor([[61., 42., 16.,  ..., 14.,  1., 22.],
        [54., 18., 12.,  ..., 11.,  0., 16.],
        [54., 61.,  9.,  ..., 10.,  0., 39.],
        ...,
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0.,  1.]])
tensor([[5.0000e+00, 1.3661e-02, 1.9170e+00, 9.2901e-01],
        [3.2000e+01, 1.0667e-01, 1.0993e+01, 9.5597e-01],
        [1.4000e+01, 4.4304e-02, 4.1508e+00, 8.7719e-01],
        ...,
        [3.0000e+00, 2.1429e-01, 1.1592e+00, 8.1650e-01],
        [3.0000e+00, 2.7273e-01, 1.1148e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 4.0825e-01]])
2
Validate:	 True
Num. nodes:	 9916
Num. edges:	 89616
Num. fea

In [17]:
# Features details
# pd.DataFrame(data.x.cpu().numpy()).describe()

### Train

In [18]:
def fit_TGAE_subgraph_(encoder, dataset, no_samples, model, epochs, train_loader, lr, test_pairs=None):
	best_hitAtOne = 0
	best_hitAtFive = 0
	best_hitAtTen = 0
	best_hitAtFifty = 0
	list_loss = []

	optimizer = Adam(model.parameters(), lr=lr,weight_decay=5e-4)
	
	# Initialize early stopping
	patience = 10
	delta = 1e-4 # 1e-4
	warmup = 10
	early_stopping = EarlyStopping(patience=patience, delta=delta, warmup=warmup, verbose=True)

	loop_obj = tqdm(range(1, epochs + 1))
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		
		# Train
		model.train()
		loss = 0.0
		
		for ts in random.sample(train_set, k=len(train_set)): # shuffle train_set
			data = train_loader[ts]

			# Encoder
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
				# z = F.normalize(z, dim=1)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)

			# Positive edges
			pos_edge_index = data.edge_index
			
			# Negative edges
			# option 1
			neg_edge_index = negative_sampling(
				edge_index=data.edge_index,
				num_nodes=z.size(0),
				num_neg_samples=pos_edge_index.size(1), # Change 2 to other value if needed
				method="sparse"
			)

			# option 2 Negative edges (dynamic)
			""" ratio = neg_ratio_schedule(epoch, epochs)
			num_neg = compute_num_neg_samples(
				edge_index=edge_index,
				num_nodes=z.size(0),
				ratio=ratio
			)
			neg_edge_index = negative_sampling(
				edge_index=edge_index,
				num_nodes=z.size(0),
				num_neg_samples=num_neg,
				method="sparse"
			) """
			
			# Decoder
			# option 1
			pos_logits = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
			neg_logits = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)
			
			# option 2
			""" pos_logits = F.cosine_similarity(
				z[pos_edge_index[0]],
				z[pos_edge_index[1]],
				dim=1
			)
			neg_logits = F.cosine_similarity(
				z[neg_edge_index[0]],
				z[neg_edge_index[1]],
				dim=1
			) """

			# Loss
			pos_labels = torch.ones_like(pos_logits)
			neg_labels = torch.zeros_like(neg_logits)

			# option 1
			""" loss_pos = binary_cross_entropy_with_logits(pos_logits, pos_labels)
			loss_neg = binary_cross_entropy_with_logits(neg_logits, neg_labels)
			loss += loss_pos + loss_neg """

			# option 2
			# num_pos = pos_edge_index.size(1)
			# num_neg = neg_edge_index.size(1)
			# pos_weight = torch.tensor([num_neg / num_pos], device=device)
			logits = torch.cat([pos_logits, neg_logits], dim=0)
			labels = torch.cat([pos_labels, neg_labels], dim=0)
			loss_temp = F.binary_cross_entropy_with_logits(logits, labels) #, pos_weight=pos_weight) # with pos_weight
			loss += loss_temp
			
		optimizer.zero_grad()
		loss = loss / no_samples
		loss.backward()
		optimizer.step()

		loop_obj.set_postfix_str(f"Loss: {loss.item():.4f}")
		list_loss.append(loss.item())

		# Check early stopping condition
		early_stopping.check_early_stop(loss.item(), epoch)
		if early_stopping.stop_training:
			print(f"Early stopping at epoch {epoch}")
			break

		# Evaluation (for firts dataset)
		""" model.eval()
		with torch.no_grad():
			keys = list(train_loader.keys())
			data1 = train_loader[keys[0]]
			data2 = train_loader[keys[1]]

			z1 = model(data1.x, data1.edge_index).detach()
			z2 = model(data2.x, data2.edge_index).detach()
			
			# Similarity matrix
			# option 1
			D = torch.cdist(z1, z2, 2)

			# option 2 (GPU problem)
			# D = 1 - F.cosine_similarity(z1.unsqueeze(1), z2.unsqueeze(0), dim=-1)

			# option 3 (Decoder cosine similarity)
			" "" z1n = F.normalize(z1, dim=1)
			z2n = F.normalize(z2, dim=1)
			D = 1 - (z1n @ z2n.T) " ""

			if dataset == "ACM_DBLP":
				test_idx = test_pairs[:, 0].astype(int)
				labels = test_pairs[:, 1].astype(int)
			else:
				test_idx = test_pairs[0, :].astype(int)
				labels = test_pairs[1, :].astype(int)
				
			hitAtOne = 0
			hitAtFive = 0
			hitAtTen = 0
			hitAtFifty = 0
			hitAtHundred = 0
			for i in range(len(test_idx)):
				dist_list = D[test_idx[i]]
				sorted_neighbors = torch.argsort(dist_list).cpu()
				label = labels[i]
				for j in range(100):
					if (sorted_neighbors[j].item() == label):
						if (j == 0):
							hitAtOne += 1
							hitAtFive += 1
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 4):
							hitAtFive += 1
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 9):
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 49):
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 100):
							hitAtHundred += 1
							break
			cur_hitAtOne = hitAtOne / len(test_idx)
			cur_hitAtFive = hitAtFive / len(test_idx)
			cur_hitAtTen = hitAtTen / len(test_idx)
			cur_hitAtFifty = hitAtFifty / len(test_idx)

			if(cur_hitAtOne > best_hitAtOne): best_hitAtOne = cur_hitAtOne
			if (cur_hitAtFive > best_hitAtFive): best_hitAtFive = cur_hitAtFive
			if (cur_hitAtTen > best_hitAtTen): best_hitAtTen = cur_hitAtTen
			if (cur_hitAtFifty > best_hitAtFifty): best_hitAtFifty = cur_hitAtFifty

	print("The best results achieved:")
	print("Hit@1: ", end="")
	print(best_hitAtOne)
	print("Hit@5: ", end="")
	print(best_hitAtFive)
	print("Hit@10: ", end="")
	print(best_hitAtTen)
	print("Hit@50: ", end="")
	print(best_hitAtFifty) """

	# Evaluation (for others dataset)
	dict_node_embeddings = {}
	model.eval()
	with torch.no_grad():
		for ts in train_set:
			data = train_loader[ts]
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)
			dict_node_embeddings[ts] = z.cpu().numpy()

	del loss, z
	# torch.cuda.synchronize()
	torch.cuda.empty_cache()
	
	return dict_node_embeddings, list_loss

In [19]:
def fit_TGAE_subgraph(model, encoder, exp, group_id, subgroups_id, train_loader, train_set, no_samples, epochs, lr):
	list_loss = []

	optimizer = Adam(model.parameters(), lr=lr, weight_decay=5e-4)
	
	# Initialize early stopping
	patience = 10
	delta = 1e-4
	warmup = 10
	early_stopping = EarlyStopping(patience=patience, delta=delta, warmup=warmup, verbose=True)

	loop_obj = tqdm(range(1, epochs + 1))
	
	# Train mode
	model.train() # Set the model to training mode (enables dropout and BN updates)
	
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		
		optimizer.zero_grad() # Reset accumulated gradients before backpropagation
		
		loss = torch.tensor(0.0, device=device)
		
		for ts in random.sample(train_set, k=len(train_set)): # shuffle train_set
			data = train_loader[ts]

			# Encoder
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
				# z = F.normalize(z, dim=1)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)

			# Positive edges
			pos_edge_index = data.edge_index
			
			# Negative edges
			# option 1
			neg_edge_index = negative_sampling(
				edge_index=data.edge_index,
				num_nodes=z.size(0),
				num_neg_samples=pos_edge_index.size(1), # Change 2 to other value if needed
				method="sparse"
			)

			# option 2 Negative edges (dynamic)
			""" ratio = neg_ratio_schedule(epoch, epochs)
			num_neg = compute_num_neg_samples(
				edge_index=edge_index,
				num_nodes=z.size(0),
				ratio=ratio
			)
			neg_edge_index = negative_sampling(
				edge_index=edge_index,
				num_nodes=z.size(0),
				num_neg_samples=num_neg,
				method="sparse"
			) """
			
			# Decoder
			# option 1
			pos_logits = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
			neg_logits = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)
			
			# option 2
			""" pos_logits = F.cosine_similarity(
				z[pos_edge_index[0]],
				z[pos_edge_index[1]],
				dim=1
			)
			neg_logits = F.cosine_similarity(
				z[neg_edge_index[0]],
				z[neg_edge_index[1]],
				dim=1
			) """

			# Loss
			pos_labels = torch.ones_like(pos_logits)
			neg_labels = torch.zeros_like(neg_logits)

			# option 1
			""" loss_pos = binary_cross_entropy_with_logits(pos_logits, pos_labels)
			loss_neg = binary_cross_entropy_with_logits(neg_logits, neg_labels)
			loss += loss_pos + loss_neg """

			# option 2
			# num_pos = pos_edge_index.size(1)
			# num_neg = neg_edge_index.size(1)
			# pos_weight = torch.tensor([num_neg / num_pos], device=device)
			logits = torch.cat([pos_logits, neg_logits], dim=0)
			labels = torch.cat([pos_labels, neg_labels], dim=0)
			loss_temp = F.binary_cross_entropy_with_logits(logits, labels) #, pos_weight=pos_weight) # with pos_weight
			loss += loss_temp
			
		loss = loss / no_samples
		loss.backward() # Compute gradients of the loss w.r.t. model parameters
		optimizer.step() # Update model parameters using computed gradients

		loop_obj.set_postfix_str(f"Loss: {loss.item():.4f}")
		list_loss.append(loss.item())

		# Check early stopping condition
		early_stopping.check_early_stop(loss.item(), epoch)
		if early_stopping.stop_training:
			print(f"Early stopping at epoch {epoch}")
			break
		# torch.cuda.empty_cache()
	# Evaluation (for others dataset)
	list_df_node_embeddings = []

	model.eval() # Set the model to evaluation mode (disables dropout and BN updates)
	with torch.no_grad():
		for subgroup_id in subgroups_id:

			# Get node embeddings
			data = train_loader[f"{group_id}_{subgroup_id}"]
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)
			
			df_node_embeddings = pd.DataFrame(z.cpu().numpy())
			df_node_embeddings["subgroup_id"] = [subgroup_id] * len(df_node_embeddings)
			
			# Get node ids to replace original Id (this is only for MS data)
			""" df_nodes = pd.read_csv(f"experiments/output/{exp}/preprocessing/graphs_data/nodes_{group_id}_{subgroup_id}.csv")
			# idx, id, mz, rt, intensity_mean, intensity_cv
			df_node_embeddings.insert(0, "Id", df_nodes["id"].values) """

			df_node_embeddings.insert(0, "Id", range(len(df_node_embeddings))) # this is only for no MS data
			list_df_node_embeddings.append(df_node_embeddings)

		# Concat node embeddings
		df_node_embeddings_concat = pd.concat(list_df_node_embeddings, ignore_index=True)
		
		# Save node embeddings
		df_node_embeddings_concat.to_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", index=False)
	
	# Save loss
	np.save(f"experiments/output/{exp}/loss/{encoder}_{group_id}.npy", np.array(list_loss))

	del loss, z
	# torch.cuda.synchronize()
	torch.cuda.empty_cache()

In [20]:
list_train_set

[['ACM-DBLP_1', 'ACM-DBLP_2']]

In [21]:
list_train_loader

[{'ACM-DBLP_1': Data(x=[9872, 17], edge_index=[2, 88994], edge_attr=[88994, 4]),
  'ACM-DBLP_2': Data(x=[9916, 17], edge_index=[2, 99532], edge_attr=[99532, 4])}]

In [22]:
list_runtime = []

for encoder in encoders:
	for i, train_loader in enumerate(list_train_loader):
		np.random.seed(0)
		torch.manual_seed(0)
		torch.cuda.manual_seed(0)
		torch.cuda.manual_seed_all(0)
		torch.backends.cudnn.deterministic = True
		torch.backends.cudnn.benchmark = False

		group_id = groups_id[i]
		subgroups_id_ = subgroups_id[group_id]

		train_set = list_train_set[i]

		no_samples = len(train_set) # * (1 + 1)  # num datasets * num of samples by dataset 
		input_dim = train_loader[train_set[0]].num_node_features

		if encoder == "GIN":
			model = TGAE_GIN(NUM_HIDDEN_LAYERS,
						input_dim,
						HIDDEN_DIM,
						output_feature_size).to(device)
		elif encoder == "GINE":
			edge_dim = train_loader[train_set[0]].edge_attr.size(1)

			model = TGAE_GINE(NUM_HIDDEN_LAYERS,
						input_dim,
						HIDDEN_DIM,
						output_feature_size, edge_dim).to(device)

		print("Fitting model")
		print(encoder, dataset, train_set, lr, epochs, input_dim, output_feature_size, no_samples)

		# Calculate elapsed time
		start_time = time.perf_counter()
		
		fit_TGAE_subgraph(model, encoder, dataset, group_id, subgroups_id_, train_loader, train_set, no_samples, epochs, lr)

		end_time = time.perf_counter()
		execution_time = end_time - start_time
		list_runtime.append(execution_time)

Fitting model
GIN exp100 ['ACM-DBLP_1', 'ACM-DBLP_2'] 0.0001 500 17 128 2


Epoch: 133:  26%|██▋       | 132/500 [01:59<05:33,  1.10it/s, Loss: 0.4052]


Stopping early as no improvement has been observed.
Early stopping at epoch 133
Fitting model
GINE exp100 ['ACM-DBLP_1', 'ACM-DBLP_2'] 0.0001 500 17 128 2


Epoch: 226:  45%|████▌     | 225/500 [04:01<04:54,  1.07s/it, Loss: 0.4393]


Stopping early as no improvement has been observed.
Early stopping at epoch 226


### Similarity analysis

In [23]:
import torch


def mrr_score(similarity: torch.Tensor,
              test_pairs: torch.Tensor,
              mode: str = 'mean') -> float:
    r"""Mean Reciprocal Rank (MRR) score of pairwise alignment results.

    Parameters
    ----------
    similarity : torch.Tensor
        Similarity matrix of shape (n1, n2) where n1 and n2 are the number of nodes in graph1 and graph2.
    test_pairs : torch.Tensor
        Test pairs of shape (m, 2) where m is the number of test pairs.
    mode : str, optional
        Mode for MRR score. Options are 'mean', 'max', 'ltr' (left-to-right), 'rtl' (right-to-left). Default is 'mean'.

    """

    if mode == 'mean':
        mrr = mrr_mean_score(similarity, test_pairs)
    elif mode == 'max':
        mrr = mrr_max_score(similarity, test_pairs)
    elif mode == 'ltr':
        mrr = mrr_ltr_score(similarity, test_pairs)
    elif mode == 'rtl':
        mrr = mrr_rtl_score(similarity, test_pairs)
    else:
        raise ValueError(f"Invalid mode: {mode}")
    return mrr


def mrr_ltr_score(similarity, test_pairs):
    r"""Mean Reciprocal Rank (MRR) score of graph1(left) to graph2(right) alignment."""
    test_pairs = test_pairs.to(similarity.device)
    ranks1 = torch.argsort(-similarity[test_pairs[:, 0]], dim=1)
    signal1_hit = ranks1 == test_pairs[:, 1].view(-1, 1)
    mrr = torch.mean(1 / (torch.where(signal1_hit)[1].float() + 1)).item()
    return mrr


def mrr_rtl_score(similarity, test_pairs):
    r"""Mean Reciprocal Rank (MRR) score of graph2(right) to graph1(left) alignment."""
    test_pairs = test_pairs.to(similarity.device)
    ranks2 = torch.argsort(-similarity.T[test_pairs[:, 1]], dim=1)
    signal2_hit = ranks2 == test_pairs[:, 0].view(-1, 1)
    mrr = torch.mean(1 / (torch.where(signal2_hit)[1].float() + 1)).item()
    return mrr


def mrr_max_score(similarity, test_pairs):
    r"""Max Mean Reciprocal Rank (MRR) score of left-to-right and right-to-left alignments."""
    mrr_ltr = mrr_ltr_score(similarity, test_pairs)
    mrr_rtl = mrr_rtl_score(similarity, test_pairs)
    mrr = max(mrr_ltr, mrr_rtl)

    return mrr


def mrr_mean_score(similarity, test_pairs):
    r"""Mean Mean Reciprocal Rank (MRR) score of left-to-right and right-to-left alignments."""
    mrr_ltr = mrr_ltr_score(similarity, test_pairs)
    mrr_rtl = mrr_rtl_score(similarity, test_pairs)
    mrr = (mrr_ltr + mrr_rtl) / 2

    return mrr

from typing import Union
import torch


def hits_ks_scores(simiarity: torch.Tensor,
                   test_pairs: torch.Tensor,
                   ks: Union[list[int], tuple[int, ...]] = (1, 10, 30, 50),
                   mode: str = 'mean') -> dict[int, float]:
    r"""Hits@K scores of pairwise alignment results.

    Parameters
    ----------
    simiarity : torch.Tensor
        Similarity matrix of shape (n1, n2) where n1 and n2 are the number of nodes in graph1 and graph2.
    test_pairs : torch.Tensor
        Test pairs of shape (m, 2) where m is the number of test pairs.
    ks : list[int] or tuple[int, ...], optional
        List of k values for Hits@K scores. Default is (1, 10, 30, 50).
    mode : str, optional
        Mode for Hits@K scores. Options are 'mean', 'max', 'ltr' (left-to-right), 'rtl' (right-to-left). Default is 'mean'.
    """

    if mode == 'mean':
        hits_ks = hits_ks_mean_scores(simiarity, test_pairs, ks=ks)
    elif mode == 'max':
        hits_ks = hits_ks_max_scores(simiarity, test_pairs, ks=ks)
    elif mode == 'ltr':
        hits_ks = hits_ks_ltr_scores(simiarity, test_pairs, ks=ks)
    elif mode == 'rtl':
        hits_ks = hits_ks_rtl_scores(simiarity, test_pairs, ks=ks)
    else:
        raise ValueError(f"Invalid mode: {mode}")
    return hits_ks


def hits_ks_ltr_scores(similarity, test_pairs, ks=None):
    r"""Hits@K scores of graph1(left) to graph2(right) alignment."""
    test_pairs = test_pairs.to(similarity.device)
    hits_ks = {}
    ranks1 = torch.argsort(-similarity[test_pairs[:, 0]], dim=1)
    signal1_hit = ranks1 == test_pairs[:, 1].view(-1, 1)
    for k in ks:
        hits_ks[k] = (torch.sum(signal1_hit[:, :k]) / test_pairs.shape[0]).item()

    return hits_ks


def hits_ks_rtl_scores(similarity, test_pairs, ks=None):
    r"""Hits@K scores of graph2(right) to graph1(left) alignment."""
    test_pairs = test_pairs.to(similarity.device)
    hits_ks = {}
    ranks2 = torch.argsort(-similarity.T[test_pairs[:, 1]], dim=1)
    signal2_hit = ranks2 == test_pairs[:, 0].view(-1, 1)
    for k in ks:
        hits_ks[k] = (torch.sum(signal2_hit[:, :k]) / test_pairs.shape[0]).item()

    return hits_ks


def hits_ks_max_scores(similarity, test_pairs, ks=None):
    r"""Max Hits@K scores of left-to-right and right-to-left alignments."""
    hits_ks = {}

    hits_ks_ltr = hits_ks_ltr_scores(similarity, test_pairs, ks=ks)
    hits_ks_rtl = hits_ks_rtl_scores(similarity, test_pairs, ks=ks)
    for k in ks:
        hits_ks[k] = max(hits_ks_ltr[k], hits_ks_rtl[k])

    return hits_ks


def hits_ks_mean_scores(similarity, test_pairs, ks=None):
    r"""Mean Hits@K scores of left-to-right and right-to-left alignments."""
    hits_ks = {}

    hits_ks_ltr = hits_ks_ltr_scores(similarity, test_pairs, ks=ks)
    hits_ks_rtl = hits_ks_rtl_scores(similarity, test_pairs, ks=ks)
    for k in ks:
        hits_ks[k] = (hits_ks_ltr[k] + hits_ks_rtl[k]) / 2

    return hits_ks


import torch
import torch.nn.functional as F


def get_normalized_neg_exp_dist(emb1, emb2, p=2, device='cpu'):
    emb1 = emb1.to(device)
    emb2 = emb2.to(device)
    normalized_emb1 = F.normalize(emb1, p=p, dim=1)
    normalized_emb2 = F.normalize(emb2, p=p, dim=1)
    return torch.exp(-(normalized_emb1 @ normalized_emb2.T))


def pairwise_cosine_similarity(x, y, p=2):
    normalized_x = F.normalize(x, p=p, dim=1)
    normalized_y = F.normalize(y, p=p, dim=1)
    return normalized_x @ normalized_y.T

In [35]:
k = 1 # Change
# list_count_meta = []

for encoder in encoders:
	for group_id in groups_id:
		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
		# Id, 0, 1,	2, ..., subgroup_id

		subgroups_id_ = subgroups_id[group_id]

		# Calculate distance matrix (KNN)
		knn = NearestNeighbors(n_neighbors=k, metric="euclidean")

		df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroups_id_[0]]
		x = df_node_embeddings.iloc[:, 1:-1].values

		df_node_alignment = pd.DataFrame()
		df_node_alignment[f"{group_id}_{subgroups_id_[0]}"] = df_node_embeddings["Id"].values

		for subgroup_id in subgroups_id_[1:]:
			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
			y = df_node_embeddings.iloc[:, 1:-1].values

			knn.fit(y)
			distances, indices = knn.kneighbors(x)
			indices = indices.squeeze() # (N,)

			df_node_alignment[f"{group_id}_{subgroup_id}"] = df_node_embeddings["Id"].values[indices]
			
			""" df_node_alignment[f"distances"] = distances
			avg_distances = df_node_alignment["distances"].mean()
			df_node_alignment = df_node_alignment[df_node_alignment["distances"] <= avg_distances].iloc[:, :-1]
			df_node_alignment """
		
		# Save previus node alignment
		df_node_alignment.to_csv(f"experiments/output/{exp}/filter_raw/node_alignment_{encoder}_{group_id}.csv")
		display(df_node_alignment)

		# Find node alignment for all datasets
		""" df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]
		list_count_meta.append([encoder, group_id, len(df_node_alignment_filter), len(df_node_alignment)])
		# print(len(df_node_alignment_filter))
		# display(df_node_alignment_filter)
		print(f"{encoder}-{group_id}: {len(df_node_alignment_filter)}")
		
		# Save common node id
		# common_node_id = df_node_alignment_filter.iloc[:, 0].values
		# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy", np.array(common_node_id)) """

		# Save common node id as .csv
		df_common_node_id = df_node_alignment.iloc[:, [0, 1]].copy() # df_common_node_id = df_node_alignment_filter.iloc[:, [0, 1]].copy()
		df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
		df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", index=False, header=False)

# Intersection and Union
""" list_common_node_id = []
for encoder in encoders:
	for group_id in groups_id:
		# Read common node
		common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		list_common_node_id.append(common_node_id)

	common_node_id_union = list(set.union(*map(set, list_common_node_id)))
	common_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))

	# Save common node id
	np.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))
	np.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection)) """

,ACM-DBLP_1,ACM-DBLP_2
0,0,6829
1,1,601
2,2,3102
3,3,3584
4,4,3744
...,...,...
9867,9867,3057
9868,9868,9011
9869,9869,2258
9870,9870,5284


,ACM-DBLP_1,ACM-DBLP_2
0,0,6829
1,1,601
2,2,3102
3,3,3584
4,4,3744
...,...,...
9867,9867,3057
9868,9868,9011
9869,9869,3145
9870,9870,7181


' list_common_node_id = []\nfor encoder in encoders:\n\tfor group_id in groups_id:\n\t\t# Read common node\n\t\tcommon_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")\n\t\tlist_common_node_id.append(common_node_id)\n\n\tcommon_node_id_union = list(set.union(*map(set, list_common_node_id)))\n\tcommon_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))\n\n\t# Save common node id\n\tnp.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))\n\tnp.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection)) '

---

In [ ]:
# Test PlanetAlign

In [54]:
list_node_embeddings = []
for encoder in encoders:
	for group_id in groups_id:
		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
		# Id, 0, 1,	2, ..., subgroup_id

		subgroups_id_ = subgroups_id[group_id]
		for subgroup_id in subgroups_id_:
			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
			list_node_embeddings.append(df_node_embeddings)

In [56]:
emb1 = list_node_embeddings[0].iloc[:, 1:-1]
emb2 = list_node_embeddings[1].iloc[:, 1:-1]
emb2

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
9872,-0.130034,0.255331,-0.239941,-0.270789,-0.016103,-0.082908,-0.072873,-0.031609,-0.228273,-0.308423,...,-0.061503,0.221828,0.208057,-0.244057,0.033628,-0.052018,-0.159795,0.128505,-0.010807,-0.066124
9873,-0.006464,0.032828,-0.080452,0.108813,-0.194687,-0.364968,-0.412754,0.057062,0.138109,-0.308810,...,-0.032540,0.213179,0.308197,-0.171641,0.188028,0.181473,0.254357,-0.285859,-0.437220,0.070326
9874,-0.127405,-0.091370,-0.144744,0.021284,0.040572,0.032154,-0.319227,-0.217123,-0.035258,-0.040149,...,-0.113006,0.084632,0.032391,0.003891,-0.214141,-0.070046,0.125312,0.186553,0.221579,0.039755
9875,0.186935,-0.611492,0.017776,0.167756,-0.093210,-0.094448,0.292002,-0.320394,0.393115,-0.079211,...,0.204952,0.357895,0.003122,0.093526,0.284539,0.022813,-0.001226,0.027653,0.148726,0.179052
9876,0.114581,-0.561822,0.355061,0.068278,-0.554124,0.139434,0.317196,-0.108242,0.034018,0.016250,...,0.058832,-0.133809,-0.037814,0.091659,-0.539355,-0.361670,-0.566977,0.045701,0.124689,0.177207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19783,-0.117728,-0.143560,0.032919,0.190201,-0.033641,-0.201288,-0.004250,-0.201676,0.030543,0.038320,...,0.065489,0.226508,0.171864,0.003100,-0.003429,-0.102059,0.073310,0.060863,-0.043461,-0.017621
19784,-0.308677,0.022118,0.072047,-0.185380,0.136058,-0.116049,-0.017335,-0.168350,0.265890,-0.120708,...,-0.293552,0.209377,-0.141928,-0.159477,0.036005,0.228347,-0.151263,-0.110595,-0.056105,-0.057551
19785,0.153999,0.092768,-0.262854,-0.325169,0.181116,0.086331,-0.124943,0.403796,-0.257302,0.085615,...,0.088653,0.215406,0.102428,-0.067404,-0.015896,0.412412,-0.269401,-0.182321,-0.118608,-0.211206
19786,0.302877,0.071467,0.088424,-0.115038,-0.154146,-0.176729,-0.166130,0.063950,-0.049999,-0.183497,...,-0.149909,-0.083526,-0.412623,-0.025336,0.070326,-0.155437,-0.190941,-0.060237,-0.135807,-0.263476


In [57]:
emb1_ = torch.tensor(emb1.values)
emb2_ = torch.tensor(emb2.values)
emb2_

tensor([[-0.1300,  0.2553, -0.2399,  ...,  0.1285, -0.0108, -0.0661],
        [-0.0065,  0.0328, -0.0805,  ..., -0.2859, -0.4372,  0.0703],
        [-0.1274, -0.0914, -0.1447,  ...,  0.1866,  0.2216,  0.0398],
        ...,
        [ 0.1540,  0.0928, -0.2629,  ..., -0.1823, -0.1186, -0.2112],
        [ 0.3029,  0.0715,  0.0884,  ..., -0.0602, -0.1358, -0.2635],
        [ 0.1508,  0.1844, -0.2160,  ..., -0.2119, -0.0735,  0.1773]],
       dtype=torch.float64)

In [58]:
test_pairs_ = torch.tensor(df_node_alignment_truth.values)
test_pairs_

tensor([[   0, 6829],
        [   2, 3102],
        [   3, 3584],
        ...,
        [ 200, 3360],
        [7605, 5020],
        [6102,  570]], dtype=torch.int32)

In [66]:
from sklearn.metrics.pairwise import cosine_similarity

S = pairwise_cosine_similarity(emb1_, emb2_)

""" S = cosine_similarity(emb1, emb2)
S = torch.tensor(S) """
S

tensor([[-1.0634e-01, -1.7257e-01, -1.0944e-01,  ..., -6.8777e-02,
         -1.5640e-01, -1.0803e-01],
        [-1.5167e-01,  2.7968e-01, -6.0520e-02,  ..., -9.7583e-02,
         -1.1844e-01, -1.5854e-01],
        [-1.3946e-01, -2.1456e-01, -1.8954e-01,  ..., -6.4481e-02,
         -1.7093e-01, -6.5602e-02],
        ...,
        [-4.4494e-02,  3.3101e-01,  2.3564e-01,  ..., -1.1105e-01,
         -7.9655e-02, -1.3459e-01],
        [ 3.0931e-01,  1.1011e-01, -7.7260e-02,  ...,  1.9007e-01,
          8.4790e-03,  2.5373e-01],
        [ 2.0878e-01, -7.1382e-02, -1.7801e-01,  ..., -8.0620e-02,
          6.5301e-02,  1.7791e-04]], dtype=torch.float64)

In [51]:
from sklearn.metrics import pairwise_distances

S = pairwise_distances(emb1, emb2, metric='euclidean')
print(S)
S = torch.tensor(S)
print(S.shape)
S

[[2.75814556 3.57757626 2.60666039 ... 3.16716121 2.25983553 2.22133458]
 [3.21841716 4.15025194 3.58234171 ... 3.40247581 3.17068123 3.18589566]
 [2.46270234 3.14671904 2.69741341 ... 2.67436454 2.35711401 2.47255099]
 ...
 [2.91580708 3.80355834 2.7979658  ... 3.19406384 2.43007875 2.46135169]
 [2.48899469 3.5859164  2.78806805 ... 2.97397832 2.36206193 2.24859874]
 [2.53178875 3.68892563 2.56029565 ... 2.7618229  2.55726176 2.52603426]]
torch.Size([9916, 9916])


tensor([[2.7581, 3.5776, 2.6067,  ..., 3.1672, 2.2598, 2.2213],
        [3.2184, 4.1503, 3.5823,  ..., 3.4025, 3.1707, 3.1859],
        [2.4627, 3.1467, 2.6974,  ..., 2.6744, 2.3571, 2.4726],
        ...,
        [2.9158, 3.8036, 2.7980,  ..., 3.1941, 2.4301, 2.4614],
        [2.4890, 3.5859, 2.7881,  ..., 2.9740, 2.3621, 2.2486],
        [2.5318, 3.6889, 2.5603,  ..., 2.7618, 2.5573, 2.5260]],
       dtype=torch.float64)

In [67]:
hits = hits_ks_scores(S, test_pairs_, mode='mean')
mrr = mrr_score(S, test_pairs_, mode='mean')
hits, mrr

({1: 0.6040316224098206,
  10: 0.8556521832942963,
  30: 0.9038735330104828,
  50: 0.9212648272514343},
 0.6902691721916199)

---

### Network alignment (metrics)

In [61]:
df_node_alignment_truth

,0,1
0,0,6829
1,2,3102
2,3,3584
3,4,3744
4,5,685
...,...,...
6320,5415,2964
6321,7191,3994
6322,200,3360
6323,7605,5020


In [62]:
list_accuracy = []
list_encoder = []

for encoder in encoders:
	for group_id in groups_id:
		# Read node alignment (after run analysis.ipynb)
		df_common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", header=None)
		# df_common_node_id[[0, 1]] = np.sort(df_common_node_id[[0, 1]], axis=1)

		# Evaluation
		df_matching = df_common_node_id.merge(df_node_alignment_truth, on=[0, 1], how="inner")

		# Accuracy
		accuracy = len(df_matching) / len(df_node_alignment_truth)
		
		list_accuracy.append(accuracy)
		list_encoder.append(encoder)

##### Summary

In [63]:
dict_summary = {
    "Dataset": [raw_data_file] * 2,
    "Model": list_encoder,
    "Accuracy": list_accuracy,
    "Runtime": list_runtime
}
df_summary = pd.DataFrame(dict_summary)
df_summary

,Dataset,Model,Accuracy,Runtime
0,ACM_DBLP,GIN,0.600000,120.981682
1,ACM_DBLP,GINE,0.610119,242.757470
